<a href="https://colab.research.google.com/github/srabani-khuntia/LLM_Using_Python/blob/main/19th_June.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import torch.nn as nn

In [7]:
class Generator(nn.Module):
  def __init__(self):
    super(Generator, self).__init__()
    self.model = nn.Sequential(
        nn.Linear(latent_dim, hidden_dim),
        nn.BatchNorm1d(hidden_dim),
        nn.LeakyReLU(0.2),

        nn.Linear(hidden_dim, hidden_dim * 2),
        nn.BatchNorm1d(hidden_dim * 2),
        nn.LeakyReLU(0.2),

        nn.Linear(hidden_dim * 2, image_dim),
        nn.Tanh()
    )

In [8]:
class Discriminator(nn.Module):
  def __init__(self):
    super(Discriminator, self).__init__()
    self.model = nn.Sequential(
        nn.Linear(image_dim, hidden_dim * 2),
        nn.LayerNorm(hidden_dim * 2),
        nn.LeakyReLU(0.2),
        nn.Dropout(0.3),
        nn.Linear(hidden_dim * 2, hidden_dim), #Input 256
        nn.LayerNorm(hidden_dim),
        nn.LeakyReLU(0.2),
        nn.Dropout(0.3),
        nn.Linear(hidden_dim, 1),
        nn.Sigmoid()
    )

In [9]:
def train_gan(generator, discriminnator, dataloader, num_epochs, device, g_optimizer, d_optimizer, criterion):
  for epoch in range(num_epochs):
    for i, (real_images, _) in enumerate(dataloader):
      batch_size = real_images.size(0)
      real_images = real_images.view(-1, image_dim).to(device)

      #Train Discriminator
      d_optimizer.zero_grad()
      output_real = discriminator(real_images).squeeze()
      d_loss_real = criterion(output_real, real_label)

      noise = torch.randn(batch_size, latent_dim).to(device)
      fake_images = generator(noise)
      output_fake = discriminator(fake_images.detach()).squeeze()
      d_loss_fake = criterion(output_fake, fake_label)

      d_loss = d_loss_real + d_loss_fake
      d_loss.backward()
      d_optimizer.step()

      #Train Generator
      g_optimizer.zero_grad()
      output_fake = discriminator(fake_images).squeeze()
      g_loss = criterion(output_fake, real_label)
      g_loss.backward()
      g_optimizer.step()